# Test Localization Pipeline

This notebook tests the full localization pipeline:
- **SequencePlaceRecognitionPipeline** for image-based place recognition
- **RansacPointCloudRegistrationPipeline** for point cloud registration
- **LocalizationPipeline** combining both

We run inference on the first 100 queries from map2 and save sample outputs.


## 1. Setup and Imports


In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn, Tensor
from torchvision import transforms as T
from PIL import Image
from tqdm.auto import tqdm

# Import from mmpr.inference (our new module)
from mmpr.inference import (
    FaissFlatIndex,
    PlaceRecognitionPipeline,
    SequencePlaceRecognitionPipeline,
    RansacPointCloudRegistrationPipeline,
    LocalizationPipeline,
    LocalizationResult,
    PointCloudStore,
)

# For loading query point clouds
from mmpr.data.pcd import SimplePCDLoader
from mmpr.data.transforms import get_T_map_to_world


/home/kartashov_ga/projects/mmpr/multimodal-place-recognition/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Configuration
ROOT_DATA_DIR = Path(
    "/mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor"
)

# Database (map1) and Query (map2) directories
DB_MAP_DIR = ROOT_DATA_DIR / "00_2023-10-25-night" / "floor_2"
QUERY_MAP_DIR = ROOT_DATA_DIR / "01_2023-11-09-twilight" / "floor_2"

# Output directory for results
OUTPUT_DIR = Path("/home/kartashov_ga/projects/mmpr/multimodal-place-recognition/experiments/localization_test_ITLP")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline parameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_QUERIES = 100  # Test on first 100 queries
MAX_WINDOW = 20    # Sequence window size
PER_FRAME_K = 10   # Top-k per frame for sequence PR
FINAL_K = 5        # Final top-k candidates for localization
VOXEL_SIZE = 0.5   # Registration voxel downsample size

# Validate paths
assert ROOT_DATA_DIR.exists(), f"Path {ROOT_DATA_DIR} does not exist"
assert DB_MAP_DIR.exists(), f"Path {DB_MAP_DIR} does not exist"
assert QUERY_MAP_DIR.exists(), f"Path {QUERY_MAP_DIR} does not exist"

print(f"Device: {DEVICE}")
print(f"DB Map: {DB_MAP_DIR}")
print(f"Query Map: {QUERY_MAP_DIR}")
print(f"Output Dir: {OUTPUT_DIR}")


Device: cuda
DB Map: /mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor/00_2023-10-25-night/floor_2
Query Map: /mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor/01_2023-11-09-twilight/floor_2
Output Dir: /home/kartashov_ga/projects/mmpr/multimodal-place-recognition/experiments/localization_test_ITLP


## 2. Load Model and Index


In [3]:
class MegaLocOPRModel(nn.Module):
    """MegaLoc model wrapper for OPR-compatible inference."""
    
    def __init__(self):
        super().__init__()
        self.model = torch.hub.load("gmberton/MegaLoc", "get_trained_model")

    def forward(self, batch: dict[str, Tensor]) -> dict[str, Tensor]:
        key = next((k for k in batch if k.startswith("images_")), None)
        if key is None:
            raise KeyError("No key starting with 'images_' found in the batch.")
        images = batch[key]
        descriptor = self.model(images)
        return {"final_descriptor": descriptor}


In [4]:
# Load the model
model = MegaLocOPRModel()
model.eval()
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Parameters: {total_params:,}")


Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Model loaded. Parameters: 228,640,321


In [5]:
# Load the FAISS index from the database
index = FaissFlatIndex.load(DB_MAP_DIR)

print(f"Index loaded from {DB_MAP_DIR}")
print(f"  - Size: {index.size()} entries")
print(f"  - Dimension: {index.dim()}")
print(f"  - Metric: {index.metric()}")


Index loaded from /mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor/00_2023-10-25-night/floor_2
  - Size: 254 entries
  - Dimension: 8448
  - Metric: l2


## 3. Create Pipelines


In [6]:
# Create Place Recognition Pipeline (single-frame)
pr_pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device=DEVICE,
)
print("PlaceRecognitionPipeline created")

# Create Sequence Place Recognition Pipeline
seq_pr_pipeline = SequencePlaceRecognitionPipeline(
    index=index,
    model=model,
    device=DEVICE,
    max_window=MAX_WINDOW,
    per_frame_k=PER_FRAME_K,
    final_k=FINAL_K,
    descriptor_agg="mean",
)
print(f"SequencePlaceRecognitionPipeline created (window={MAX_WINDOW}, per_frame_k={PER_FRAME_K}, final_k={FINAL_K})")

# Create Registration Pipeline
reg_pipeline = RansacPointCloudRegistrationPipeline(
    voxel_downsample_size=VOXEL_SIZE,
)
print(f"RansacPointCloudRegistrationPipeline created (voxel_size={VOXEL_SIZE})")

# Create full Localization Pipeline
# Note: LocalizationPipeline uses PlaceRecognitionPipeline, but works with SequencePR too
loc_pipeline = LocalizationPipeline(
    index=index,
    place_recognition=pr_pipeline,  # Using single-frame PR for now
    registration=reg_pipeline,
    index_root=DB_MAP_DIR,
    require_db_pointcloud=False,  # Skip candidates without point clouds
)
print("LocalizationPipeline created")


PlaceRecognitionPipeline created
SequencePlaceRecognitionPipeline created (window=20, per_frame_k=10, final_k=5)
RansacPointCloudRegistrationPipeline created (voxel_size=0.5)
LocalizationPipeline created


## 4. Load Query Data


In [7]:
# Image preprocessing for MegaLoc
image_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True),
])


def read_image(image_path: str | Path) -> Tensor:
    """Load and preprocess an image."""
    image = Image.open(image_path)
    image = image_transform(image)
    return image


def to_batch(image: Tensor) -> dict[str, Tensor]:
    """Convert image tensor to batch dict for model input."""
    return {"images_0": image.unsqueeze(0)}


In [8]:
# Load query point clouds using SimplePCDLoader
#T_map_to_world = get_T_map_to_world("map2")
query_pcd_loader = SimplePCDLoader(
    map_root=QUERY_MAP_DIR,
    scans_subdir="lidar",
    header=1
    #T_map_to_world=T_map_to_world,
)
print(f"Query PCD loader: {len(query_pcd_loader)} scans available")

# Get image directory
query_images_dir = QUERY_MAP_DIR / "front_cam"
if not query_images_dir.exists():
    # Try alternative paths
    for alt in ["images", "rgb"]:
        alt_dir = QUERY_MAP_DIR / alt
        if alt_dir.exists():
            query_images_dir = alt_dir
            break

print(f"Query images directory: {query_images_dir}")
assert query_images_dir.exists(), f"Images directory not found: {query_images_dir}"

# Limit to NUM_QUERIES
num_available = min(len(query_pcd_loader), NUM_QUERIES)
print(f"Will process {num_available} queries")


Query PCD loader: 249 scans available
Query images directory: /mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor/01_2023-11-09-twilight/floor_2/front_cam
Will process 100 queries


## 5. Run Inference


In [10]:
def load_trajectory_df(traj_path, sep=","):
    traj_df = pd.read_table(
        traj_path, header=1, sep=sep,
        names=["timestamp", "front_cam_ts", "back_cam_ts", "lidar_ts", "x", "y", "z", "qx", "qy", "qz", "qw"],
        comment='#'
    )
    print(f"Loaded frames with {len(traj_df)} entries.")
    return traj_df
db_map_df = load_trajectory_df(DB_MAP_DIR / "poses.csv")
db_q_df = load_trajectory_df(QUERY_MAP_DIR / "poses.csv")

# Run localization on first NUM_QUERIES frames
results: list[LocalizationResult] = []
timing_stats: list[dict] = []
errors: list[dict] = []

print(f"Running localization on {num_available} queries...")
print(f"Using: LocalizationPipeline with PlaceRecognitionPipeline + RANSAC Registration")

for qi in tqdm(range(num_available), desc="Localizing"):
    #try:
    # Load query image
    image_path = query_images_dir / f"{qi:06d}.jpg"
    if not image_path.exists():
        # Try png
        image_path = query_images_dir / f"{db_q_df.iloc[qi, 1]}.png"
    
    if not image_path.exists():
        errors.append({"query_idx": qi, "error": f"Image not found: {image_path}"})
        continue
    
    image = read_image(image_path)
    pr_input = to_batch(image)
    pr_input = {k: v.to(DEVICE) for k, v in pr_input.items()}
    
    # Load query point cloud
    query_points, query_pose7, query_pcd_path = query_pcd_loader[qi]
    query_pc = torch.from_numpy(query_points).float()
    
    # Run localization
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t_start = time.perf_counter()
    
    result = loc_pipeline.infer(
        pr_input=pr_input,
        query_pc=query_pc,
        k=FINAL_K,
    )
    
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t_end = time.perf_counter()
    
    results.append(result)
    timing_stats.append({
        "query_idx": qi,
        "time_ms": (t_end - t_start) * 1000,
        "num_candidates": len(result.candidates),
        "chosen_idx": result.chosen_idx,
    })
        
    # except Exception as e:
    #     errors.append({"query_idx": qi, "error": str(e)})
    #     continue

print(f"\nCompleted: {len(results)} successful, {len(errors)} errors")
print(errors)


Loaded frames with 254 entries.
Loaded frames with 248 entries.
Running localization on 100 queries...
Using: LocalizationPipeline with PlaceRecognitionPipeline + RANSAC Registration


Localizing:   0%|          | 0/100 [00:00<?, ?it/s]


FileNotFoundError: Point cloud file not found: /mnt/external_usb_hdd/6YL/Datasets/archive/itlp_campus_indoor/00_2023-10-25-night/floor_2/scans/000000.pcd

In [ ]:
# Timing statistics
if timing_stats:
    times = [s["time_ms"] for s in timing_stats]
    print(f"Timing Statistics (ms):")
    print(f"  Mean: {np.mean(times):.2f}")
    print(f"  Std:  {np.std(times):.2f}")
    print(f"  Min:  {np.min(times):.2f}")
    print(f"  Max:  {np.max(times):.2f}")
    print(f"  Median: {np.median(times):.2f}")


## 6. Save Results


In [ ]:
# Create output subdirectory for this run
run_dir = OUTPUT_DIR / f"run_{time.strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)
results_dir = run_dir / "localization_results"
results_dir.mkdir(exist_ok=True)

# Save individual LocalizationResults as JSON
print(f"Saving {len(results)} localization results to {results_dir}")
for i, result in enumerate(results):
    result_path = results_dir / f"result_{i:04d}.json"
    result.save(result_path)

# Save timing statistics
timing_df = pd.DataFrame(timing_stats)
timing_path = run_dir / "timing_stats.csv"
timing_df.to_csv(timing_path, index=False)
print(f"Timing stats saved to {timing_path}")

# Save errors
if errors:
    errors_path = run_dir / "errors.json"
    errors_path.write_text(json.dumps(errors, indent=2))
    print(f"Errors saved to {errors_path}")

# Save summary
summary = {
    "num_queries": num_available,
    "num_successful": len(results),
    "num_errors": len(errors),
    "config": {
        "device": DEVICE,
        "max_window": MAX_WINDOW,
        "per_frame_k": PER_FRAME_K,
        "final_k": FINAL_K,
        "voxel_size": VOXEL_SIZE,
    },
    "timing_ms": {
        "mean": float(np.mean(times)) if timing_stats else None,
        "std": float(np.std(times)) if timing_stats else None,
        "min": float(np.min(times)) if timing_stats else None,
        "max": float(np.max(times)) if timing_stats else None,
        "median": float(np.median(times)) if timing_stats else None,
    },
    "db_map": str(DB_MAP_DIR),
    "query_map": str(QUERY_MAP_DIR),
}
summary_path = run_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {summary_path}")

print(f"\n=== All outputs saved to {run_dir} ===")


In [ ]:
# Verify we can load the saved results back
if results:
    sample_path = results_dir / "result_0000.json"
    loaded_result = LocalizationResult.load(sample_path)
    
    print("Sample loaded result:")
    print(f"  Version: {loaded_result.version}")
    print(f"  Chosen idx: {loaded_result.chosen_idx}")
    print(f"  Num candidates: {len(loaded_result.candidates)}")
    
    if loaded_result.candidates:
        c = loaded_result.candidates[0]
        print(f"  First candidate:")
        print(f"    - idx: {c.idx}")
        print(f"    - pr_distance: {c.pr_distance:.4f}")
        print(f"    - db_pose: {c.db_pose}")
        print(f"    - estimated_pose: {c.estimated_pose}")
        print(f"    - registration_confidence: {c.registration_confidence}")


## 7. (Optional) Test Sequence Place Recognition Separately


In [ ]:
# Test the SequencePlaceRecognitionPipeline separately
print("Testing SequencePlaceRecognitionPipeline on first 10 frames...")

# Reset sequence state
seq_pr_pipeline.reset()

seq_results = []
for qi in range(min(10, num_available)):
    image_path = query_images_dir / f"{qi:06d}.jpg"
    if not image_path.exists():
        image_path = query_images_dir / f"{qi:06d}.png"
    
    if image_path.exists():
        image = read_image(image_path)
        pr_input = to_batch(image)
        pr_input = {k: v.to(DEVICE) for k, v in pr_input.items()}
        
        result = seq_pr_pipeline.infer(input_frame=pr_input, k=5)
        seq_results.append(result)
        
        print(f"Frame {qi}: top-5 db_idx = {result.db_idx.tolist() if result.db_idx is not None else 'N/A'}")

# Save sample sequence PR result
if seq_results:
    sample_pr_path = run_dir / "sample_sequence_pr_result.json"
    seq_results[-1].save(sample_pr_path)
    print(f"\nSample SequencePR result saved to {sample_pr_path}")


In [ ]:
print("=" * 60)
print("LOCALIZATION PIPELINE TEST COMPLETE")
print("=" * 60)
print(f"Results directory: {run_dir}")
print(f"Total queries processed: {len(results)}")
print(f"Errors encountered: {len(errors)}")
